In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
FEATURES = ['HUFL', 'HULL', 'MUFL', 'MULL', 'LUFL', 'LULL', 'OT']
TARGET = 'OT'
LOOKBACK, HORIZON, BATCH_SIZE = 96, 4, 64
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

In [ ]:
from google.colab import drive
from pathlib import Path
import pandas as pd

drive.mount('/content/drive')
DATA_PATH = Path('/content/drive/MyDrive/CBNU/어프렌티스 프로젝트/ETTm1.csv')

df = pd.read_csv(DATA_PATH, parse_dates=['date']).sort_values('date').reset_index(drop=True)
print('Shape:', df.shape)
display(df.head())

In [ ]:
print('기간:', df.date.min(), '~', df.date.max())
print('결측치:\n', df.isna().sum())
print('중복 시각:', df.date.duplicated().sum())
print('시간 간격:\n', df.date.diff().dropna().value_counts().head())

display(df[FEATURES].describe().round(3))
assert not df[FEATURES].isna().any().any()
assert not df.date.duplicated().any()
assert (df.date.diff().dropna() == pd.Timedelta(minutes=15)).all()

In [ ]:
plt.figure(figsize=(14,4))
plt.plot(df.date, df.OT, linewidth=.5)
plt.title('ETTm1 Oil Temperature (Full Period)')
plt.xlabel('Date'); plt.ylabel('OT (°C)'); plt.tight_layout(); plt.show()

plt.figure(figsize=(14,4))
plt.plot(df.date.iloc[:7*96], df.OT.iloc[:7*96])
plt.title('ETTm1 OT (First 7 Days)')
plt.xlabel('Date'); plt.ylabel('OT (°C)'); plt.tight_layout(); plt.show()

In [ ]:
n = len(df)
train_end, val_end = int(n*.6), int(n*.8)
splits = {'train': df.iloc[:train_end].copy(),
          'val': df.iloc[train_end:val_end].copy(),
          'test': df.iloc[val_end:].copy()}
summary = pd.DataFrame([{'split':k,'rows':len(v),'start':v.date.iloc[0],
                         'end':v.date.iloc[-1],'OT_mean':v.OT.mean()}
                        for k,v in splits.items()])
display(summary)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

fig, ax = plt.subplots(figsize=(15, 5))

settings = [
    ('train', 'Train (60%)', '#2563EB'),
    ('val',   'Validation (20%)', '#F59E0B'),
    ('test',  'Test (20%)', '#16A34A')
]

for key, label, color in settings:
    part = splits[key]


    ax.plot(
        part['date'], part['OT'],
        color=color, linewidth=0.6, label=label
    )

    ax.axvspan(
        part['date'].iloc[0],
        part['date'].iloc[-1],
        color=color, alpha=0.07
    )

for key in ['val', 'test']:
    ax.axvline(
        splits[key]['date'].iloc[0],
        color='gray', linestyle='--', linewidth=1
    )

ax.set_title('ETTm1 Oil Temperature: Train / Validation / Test')
ax.set_xlabel('Date')
ax.set_ylabel('OT (°C)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.grid(alpha=0.2)
ax.legend(loc='upper right')

fig.tight_layout()
plt.show()

# 실제 분할 기간 확인
for key, label, _ in settings:
    part = splits[key]
    print(
        f"{label}: "
        f"{part['date'].iloc[0]} ~ {part['date'].iloc[-1]}"
        f" / {len(part):,}개"
    )

In [ ]:
scaler = StandardScaler()
scaler.fit(splits['train'][FEATURES])
scaled = {k: scaler.transform(v[FEATURES]).astype(np.float32)
          for k,v in splits.items()}
target_idx = FEATURES.index(TARGET)
print('Train 정규화 평균:', np.round(scaled['train'].mean(axis=0), 4))
print('Train 정규화 표준편차:', np.round(scaled['train'].std(axis=0), 4))

In [ ]:
class ETTWindowDataset(Dataset):
    def __init__(self, array, lookback=96, horizon=4, target_idx=6):
        self.array = torch.from_numpy(array)
        self.lookback, self.horizon = lookback, horizon
        self.target_idx = target_idx
        self.length = max(0, len(array)-lookback-horizon+1)
    def __len__(self):
        return self.length
    def __getitem__(self, idx):
        x = self.array[idx:idx+self.lookback]
        y = self.array[idx+self.lookback:idx+self.lookback+self.horizon, self.target_idx]
        return x, y

datasets = {k: ETTWindowDataset(v, LOOKBACK, HORIZON, target_idx)
            for k,v in scaled.items()}
train_loader = DataLoader(datasets['train'], batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(datasets['val'], batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(datasets['test'], batch_size=BATCH_SIZE, shuffle=False)
print('윈도우 개수:', {k:len(v) for k,v in datasets.items()})
for name, loader in [('train',train_loader),('val',val_loader),('test',test_loader)]:
    x,y = next(iter(loader))
    print(f'{name}: X={tuple(x.shape)}, Y={tuple(y.shape)}')

In [ ]:
def inverse_ot(y_scaled):
    return np.asarray(y_scaled) * scaler.scale_[target_idx] + scaler.mean_[target_idx]

x0,y0 = datasets['train'][0]
print('X 기간:', splits['train'].date.iloc[0], '~', splits['train'].date.iloc[LOOKBACK-1])
print('Y 기간:', splits['train'].date.iloc[LOOKBACK], '~', splits['train'].date.iloc[LOOKBACK+HORIZON-1])
print('X shape:',tuple(x0.shape),'Y shape:',tuple(y0.shape))
print('Y 실제 OT:', np.round(inverse_ot(y0.numpy()),3))
print('2단계에서 사용할 객체: train_loader, val_loader, test_loader, inverse_ot, DEVICE')

In [ ]:
import copy, time, os
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

OUT_DIR = Path('/content/drive/MyDrive/CBNU/어프렌티스 프로젝트/ettm1_results')
OUT_DIR.mkdir(parents=True, exist_ok=True)
INPUT_SIZE = len(FEATURES)
HIDDEN_SIZE = 64
NUM_LAYERS = 2
DROPOUT = 0.2
LR = 0.001
MAX_EPOCHS = 30
PATIENCE = 5
MIN_DELTA = 1e-5
print('DEVICE:', DEVICE, '| Input:', INPUT_SIZE, '| Horizon:', HORIZON)


In [ ]:
class ForecastRNN(torch.nn.Module):
    def __init__(self, kind, input_size=7, hidden_size=64, num_layers=2, dropout=0.2, horizon=4):
        super().__init__()
        modules = {'RNN': torch.nn.RNN, 'LSTM': torch.nn.LSTM, 'GRU': torch.nn.GRU}
        if kind not in modules:
            raise ValueError(f'지원하지 않는 모델: {kind}')
        self.kind = kind
        self.encoder = modules[kind](
            input_size=input_size, hidden_size=hidden_size,
            num_layers=num_layers, dropout=dropout if num_layers > 1 else 0,
            batch_first=True
        )
        self.head = torch.nn.Linear(hidden_size, horizon)

    def forward(self, x):
        outputs, _ = self.encoder(x)
        return self.head(outputs[:, -1, :])

for kind in ('RNN', 'LSTM', 'GRU'):
    m = ForecastRNN(kind, INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS, DROPOUT, HORIZON)
    print(kind, '파라미터 수:', sum(p.numel() for p in m.parameters()))


In [ ]:

from torch.utils.data import DataLoader, Subset
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd
import torch

TEST_LOOKBACK = 96
COMMON_LOOKBACK = 192

full_test_ds = ETTWindowDataset(
    scaled['test'],
    lookback=TEST_LOOKBACK,
    horizon=HORIZON,
    target_idx=target_idx
)

offset = COMMON_LOOKBACK - TEST_LOOKBACK
common_test_ds = Subset(
    full_test_ds,
    range(offset, len(full_test_ds))
)

eval_loaders = {
    '기존 전체 범위': DataLoader(
        full_test_ds, batch_size=BATCH_SIZE, shuffle=False
    ),
    '공통 범위': DataLoader(
        common_test_ds, batch_size=BATCH_SIZE, shuffle=False
    )
}

print('평가 윈도우 수')
for name, loader in eval_loaders.items():
    print(f'{name}: {len(loader.dataset):,}개')


@torch.no_grad()
def evaluate_saved_model(model, loader):
    model.eval()
    predictions, actuals = [], []

    for x, y in loader:
        predictions.append(
            model(x.to(DEVICE)).cpu().numpy()
        )
        actuals.append(y.numpy())

    pred = inverse_ot(np.concatenate(predictions))
    true = inverse_ot(np.concatenate(actuals))

    return np.sqrt(
        mean_squared_error(true.ravel(), pred.ravel())
    )



for kind in ['RNN', 'LSTM', 'GRU']:
    checkpoint = OUT_DIR / f'{kind.lower()}_best.pt'
    if not checkpoint.exists():
        raise FileNotFoundError(
            f'최초 모델 파일이 없습니다: {checkpoint}'
        )

rows = []

for kind in ['RNN', 'LSTM', 'GRU']:
    model = ForecastRNN(
        kind,
        INPUT_SIZE,
        HIDDEN_SIZE,
        NUM_LAYERS,
        DROPOUT,
        HORIZON
    ).to(DEVICE)

    checkpoint = OUT_DIR / f'{kind.lower()}_best.pt'
    state = torch.load(
        checkpoint,
        map_location=DEVICE,
        weights_only=True
    )
    model.load_state_dict(state)

    for scope, loader in eval_loaders.items():
        rows.append({
            '모델': kind,
            '평가 범위': scope,
            'RMSE': evaluate_saved_model(model, loader)
        })

    del model


for scope, loader in eval_loaders.items():
    predictions, actuals = [], []

    for x, y in loader:
        last_value = x[:, -1, target_idx].numpy()
        predictions.append(
            np.repeat(last_value[:, None], HORIZON, axis=1)
        )
        actuals.append(y.numpy())

    pred = inverse_ot(np.concatenate(predictions))
    true = inverse_ot(np.concatenate(actuals))

    rows.append({
        '모델': 'Last Value',
        '평가 범위': scope,
        'RMSE': np.sqrt(
            mean_squared_error(true.ravel(), pred.ravel())
        )
    })

result = pd.DataFrame(rows)

comparison = result.pivot(
    index='모델',
    columns='평가 범위',
    values='RMSE'
).reindex(['RNN', 'LSTM', 'GRU', 'Last Value'])

comparison['범위 변경에 따른 차이'] = (
    comparison['공통 범위'] - comparison['기존 전체 범위']
)

display(comparison.round(6))

print('\n공통 범위에서 최초 저장 모델의 순위')
display(
    result[
        (result['평가 범위'] == '공통 범위') &
        (result['모델'] != 'Last Value')
    ].sort_values('RMSE').reset_index(drop=True)
)

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)
    total_loss, total_count = 0.0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        if is_train:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(is_train):
            pred = model(x)
            loss = criterion(pred, y)
            if is_train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
        total_loss += loss.item() * x.size(0)
        total_count += x.size(0)
    return total_loss / total_count


def train_one_model(kind):
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    model = ForecastRNN(kind, INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS, DROPOUT, HORIZON).to(DEVICE)
    criterion = torch.nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    best_loss, best_epoch, best_weights, wait = float('inf'), 0, None, 0
    history = {'train': [], 'val': []}
    start = time.perf_counter()
    for epoch in range(1, MAX_EPOCHS + 1):
        train_loss = run_epoch(model, train_loader, criterion, optimizer)
        val_loss = run_epoch(model, val_loader, criterion)
        history['train'].append(train_loss)
        history['val'].append(val_loss)
        print(f'{kind:4s} | Epoch {epoch:02d} | Train {train_loss:.5f} | Val {val_loss:.5f}')
        if val_loss < best_loss - MIN_DELTA:
            best_loss = val_loss
            best_epoch = epoch
            best_weights = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
            if wait >= PATIENCE:
                print(f'{kind}: Early Stopping (best epoch {best_epoch})')
                break
    elapsed = time.perf_counter() - start
    model.load_state_dict(best_weights)
    torch.save(model.state_dict(), OUT_DIR / f'{kind.lower()}_best.pt')
    return model, history, best_epoch, best_loss, elapsed


In [ ]:
models, histories, training_info = {}, {}, []
for kind in ('RNN', 'LSTM', 'GRU'):
    model, history, best_epoch, best_val, elapsed = train_one_model(kind)
    models[kind] = model
    histories[kind] = history
    training_info.append({'Model': kind, 'Best Epoch': best_epoch,
                          'Best Val MSE (scaled)': best_val,
                          'Train Seconds': elapsed})
train_summary = pd.DataFrame(training_info)
display(train_summary.round(4))
train_summary.to_csv(OUT_DIR / 'training_summary.csv', index=False)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for kind, h in histories.items():
    axes[0].plot(range(1, len(h['train'])+1), h['train'], label=kind)
    axes[1].plot(range(1, len(h['val'])+1), h['val'], label=kind)
for ax, title in zip(axes, ['Training Loss', 'Validation Loss']):
    ax.set_title(title + ' (scaled MSE)')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE')
    ax.grid(alpha=.3)
    ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / 'loss_comparison.png', dpi=160)
plt.show()


In [ ]:
@torch.no_grad()
def predict_all(model, loader):
    model.eval()
    pred_list, true_list = [], []
    for x, y in loader:
        pred_list.append(model(x.to(DEVICE)).cpu().numpy())
        true_list.append(y.numpy())
    return np.concatenate(pred_list), np.concatenate(true_list)

results, test_predictions = [], {}
for kind, model in models.items():
    pred_scaled, true_scaled = predict_all(model, test_loader)
    pred_c = inverse_ot(pred_scaled)
    true_c = inverse_ot(true_scaled)
    test_predictions[kind] = pred_c
    mae = mean_absolute_error(true_c.ravel(), pred_c.ravel())
    mse = mean_squared_error(true_c.ravel(), pred_c.ravel())
    rmse = np.sqrt(mse)
    r2 = r2_score(true_c.ravel(), pred_c.ravel())
    results.append({'Model': kind, 'MAE (°C)': mae, 'MSE (°C²)': mse,
                    'RMSE (°C)': rmse, 'R2': r2})


last_ot_scaled = np.array([test_loader.dataset[i][0][-1, target_idx].item()
                           for i in range(len(test_loader.dataset))], dtype=np.float32)
naive_scaled = np.repeat(last_ot_scaled[:, None], HORIZON, axis=1)
naive_c = inverse_ot(naive_scaled)
true_c = inverse_ot(predict_all(models['RNN'], test_loader)[1])
naive_mse = mean_squared_error(true_c.ravel(), naive_c.ravel())
results.append({'Model': 'Last Value',
                'MAE (°C)': mean_absolute_error(true_c.ravel(), naive_c.ravel()),
                'MSE (°C²)': naive_mse, 'RMSE (°C)': np.sqrt(naive_mse),
                'R2': r2_score(true_c.ravel(), naive_c.ravel())})

metrics_df = pd.DataFrame(results).sort_values('RMSE (°C)').reset_index(drop=True)
display(metrics_df.round(4))
metrics_df.to_csv(OUT_DIR / 'test_metrics.csv', index=False)


In [ ]:
N_SHOW = min(96, len(true_c))
times = splits['test'].date.iloc[LOOKBACK + HORIZON - 1:
                                    LOOKBACK + HORIZON - 1 + N_SHOW]
plt.figure(figsize=(14, 5))
plt.plot(times, true_c[:N_SHOW, -1], label='Actual OT', linewidth=2)
for kind in ('RNN', 'LSTM', 'GRU'):
    plt.plot(times, test_predictions[kind][:N_SHOW, -1], label=kind, alpha=.8)
plt.title('ETTm1: 60-minute-ahead OT forecast (Test)')
plt.xlabel('Time')
plt.ylabel('OT (°C)')
plt.xticks(rotation=30)
plt.grid(alpha=.25)
plt.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / 'test_prediction_60min.png', dpi=160)
plt.show()
print('저장 폴더:', OUT_DIR)


In [ ]:
from google.colab import drive
from pathlib import Path
import random, copy, time, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


ABL_DIR = OUT_DIR / 'ablation'
ABL_DIR.mkdir(parents=True, exist_ok=True)
LOOKBACKS = [48, 96, 192]
MODEL_KINDS = ['RNN', 'LSTM', 'GRU']
HORIZON_ABL = 4
BATCH_ABL = 64
HIDDEN_ABL = 64
LAYERS_ABL = 2
DROPOUT_ABL = 0.2
LR_ABL = 0.001
MAX_EPOCHS_ABL = 30
PATIENCE_ABL = 5
MIN_DELTA_ABL = 1e-5
SEED_ABL = 42

required = ['scaled', 'splits', 'FEATURES', 'target_idx', 'inverse_ot', 'ETTWindowDataset', 'ForecastRNN', 'DEVICE']
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError('1~2단계의 셀을 먼저 실행해야 합니다. 누락된 객체: ' + ', '.join(missing))
print('저장 폴더:', ABL_DIR)
print('DEVICE:', DEVICE)


In [ ]:
MAX_LOOKBACK = max(LOOKBACKS)
loaders_by_lb = {}
for lb in LOOKBACKS:
    loaders = {}
    for part in ['train', 'val', 'test']:
        ds = ETTWindowDataset(scaled[part], lookback=lb, horizon=HORIZON_ABL, target_idx=target_idx)

        offset = MAX_LOOKBACK - lb
        indices = range(offset, len(ds))
        subset = Subset(ds, indices)
        loaders[part] = DataLoader(subset, batch_size=BATCH_ABL, shuffle=(part == 'train'), num_workers=0)
    loaders_by_lb[lb] = loaders
    print(f'Lookback={lb:3d} | train={len(loaders["train"].dataset):,} | val={len(loaders["val"].dataset):,} | test={len(loaders["test"].dataset):,}')
    x,y = next(iter(loaders['test']))
    print('  Test 배치:',tuple(x.shape),tuple(y.shape))
assert len({len(loaders_by_lb[lb]['test'].dataset) for lb in LOOKBACKS}) == 1


In [ ]:
def seed_everything(seed=SEED_ABL):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def epoch_loss(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total, count = 0., 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            pred = model(x)
            loss = torch.nn.functional.mse_loss(pred, y)
            if training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
        total += loss.item()*len(x)
        count += len(x)
    return total/count

def fit_ablation(kind, lb, loaders):
    seed_everything()
    model = ForecastRNN(kind, len(FEATURES), HIDDEN_ABL, LAYERS_ABL, DROPOUT_ABL, HORIZON_ABL).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR_ABL)
    best_val, best_epoch, best_state, patience = float('inf'), 0, None, 0
    history = []
    start = time.perf_counter()
    for ep in range(1, MAX_EPOCHS_ABL+1):
        train_mse = epoch_loss(model, loaders['train'], optimizer)
        val_mse = epoch_loss(model, loaders['val'])
        history.append({'Epoch': ep, 'Train MSE (scaled)': train_mse, 'Val MSE (scaled)': val_mse})
        print(f'{kind:4s} | L={lb:3d} | Epoch {ep:02d} | Train {train_mse:.5f} | Val {val_mse:.5f}')
        if val_mse < best_val - MIN_DELTA_ABL:
            best_val, best_epoch, best_state, patience = val_mse, ep, copy.deepcopy(model.state_dict()), 0
        else:
            patience += 1
            if patience >= PATIENCE_ABL:
                break
    elapsed = time.perf_counter()-start
    model.load_state_dict(best_state)
    torch.save(model.state_dict(), ABL_DIR/f'{kind.lower()}_lookback_{lb}_best.pt')
    pd.DataFrame(history).to_csv(ABL_DIR/f'{kind.lower()}_lookback_{lb}_history.csv', index=False)
    return model, best_epoch, best_val, elapsed, pd.DataFrame(history)

@torch.no_grad()
def predict_ablation(model, loader):
    model.eval()
    preds, trues = [], []
    for x,y in loader:
        preds.append(model(x.to(DEVICE)).cpu().numpy())
        trues.append(y.numpy())
    return inverse_ot(np.concatenate(preds)), inverse_ot(np.concatenate(trues))

def metrics(y, pred):
    return {'MAE (°C)':mean_absolute_error(y.ravel(),pred.ravel()),
            'MSE (°C²)':mean_squared_error(y.ravel(),pred.ravel()),
            'RMSE (°C)':np.sqrt(mean_squared_error(y.ravel(),pred.ravel())),
            'R2':r2_score(y.ravel(),pred.ravel()),
            '60min RMSE (°C)':np.sqrt(mean_squared_error(y[:,-1],pred[:,-1]))}


In [ ]:
ablation_rows = []
all_histories = {}
for lb in LOOKBACKS:
    for kind in MODEL_KINDS:
        print(f'\n========== {kind} / Lookback {lb} ==========' )
        model, best_ep, best_val, seconds, hist = fit_ablation(kind, lb, loaders_by_lb[lb])
        pred_c, true_c = predict_ablation(model, loaders_by_lb[lb]['test'])
        row = {'Model':kind, 'Lookback':lb, 'Best Epoch':best_ep,
               'Best Val MSE (scaled)':best_val, 'Train Seconds':seconds, **metrics(true_c,pred_c)}
        ablation_rows.append(row)
        all_histories[(kind,lb)] = hist
        pd.DataFrame(ablation_rows).to_csv(ABL_DIR/'ablation_results_partial.csv', index=False)
        print('Test RMSE (°C):',round(row['RMSE (°C)'],4))
        del model
        if torch.cuda.is_available(): torch.cuda.empty_cache()

ablation_df = pd.DataFrame(ablation_rows).sort_values(['Model','Lookback']).reset_index(drop=True)
display(ablation_df.round(4))
ablation_df.to_csv(ABL_DIR/'ablation_results.csv',index=False)


In [ ]:

base_loader = loaders_by_lb[MAX_LOOKBACK]['test']
naive_pred, actual = [], []
for x,y in base_loader:
    naive_pred.append(x[:,-1,target_idx].numpy()[:,None].repeat(HORIZON_ABL,axis=1))
    actual.append(y.numpy())
naive_c = inverse_ot(np.concatenate(naive_pred))
actual_c = inverse_ot(np.concatenate(actual))
naive_metrics = metrics(actual_c, naive_c)
print('Last Value 기준선:', {k:round(v,4) for k,v in naive_metrics.items()})
print('최저 RMSE 실험:')
display(ablation_df.nsmallest(3,'RMSE (°C)').round(4))
pd.DataFrame([{'Model':'Last Value',**naive_metrics}]).to_csv(ABL_DIR/'baseline_last_value.csv',index=False)


In [ ]:
pivot = ablation_df.pivot(index='Model',columns='Lookback',values='RMSE (°C)').reindex(MODEL_KINDS)
display(pivot.round(4))
pivot.to_csv(ABL_DIR/'ablation_rmse_pivot.csv')
fig, ax = plt.subplots(figsize=(8,5))
for kind in MODEL_KINDS:
    row = ablation_df[ablation_df.Model==kind].sort_values('Lookback')
    ax.plot(row.Lookback,row['RMSE (°C)'],marker='o',label=kind)
ax.axhline(naive_metrics['RMSE (°C)'],linestyle='--',color='gray',label='Last Value')
ax.set_xticks(LOOKBACKS)
ax.set_xlabel('Lookback (15-minute steps)')
ax.set_ylabel('Test RMSE (°C) — 4-step horizon')
ax.set_title('ETTm1 Lookback Ablation: RNN vs LSTM vs GRU')
ax.grid(alpha=.25)
ax.legend()
fig.tight_layout()
fig.savefig(ABL_DIR/'ablation_rmse.png',dpi=180)
plt.show()


In [ ]:
fig, axes = plt.subplots(1,3,figsize=(16,4),sharey=True)
for ax,kind in zip(axes,MODEL_KINDS):
    for lb in LOOKBACKS:
        hist = all_histories[(kind,lb)]
        ax.plot(hist['Epoch'],hist['Val MSE (scaled)'],label=f'L={lb}')
    ax.set_title(kind)
    ax.set_xlabel('Epoch')
    ax.grid(alpha=.25)
    ax.legend()
axes[0].set_ylabel('Validation MSE (scaled)')
fig.tight_layout()
fig.savefig(ABL_DIR/'ablation_val_loss.png',dpi=180)
plt.show()

config = {'lookbacks':LOOKBACKS,'horizon':HORIZON_ABL,'models':MODEL_KINDS,
          'seed':SEED_ABL,'hidden_size':HIDDEN_ABL,'num_layers':LAYERS_ABL,
          'dropout':DROPOUT_ABL,'learning_rate':LR_ABL,'batch_size':BATCH_ABL,
          'max_epochs':MAX_EPOCHS_ABL,'patience':PATIENCE_ABL,
          'comparison':'same target timestamps within each split; four horizon points evaluated'}
(ABL_DIR/'ablation_config.json').write_text(json.dumps(config,ensure_ascii=False,indent=2),encoding='utf-8')
print('저장된 결과 파일:')
for p in sorted(ABL_DIR.iterdir()): print(' -',p.name)


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

if not Path('/content/drive/MyDrive').exists():
    from google.colab import drive
    drive.mount('/content/drive')

RESULT_DIR = Path('/content/drive/MyDrive/CBNU/어프렌티스 프로젝트/ettm1_results')
ABL_DIR = RESULT_DIR / 'ablation'
FINAL_DIR = RESULT_DIR / 'final_analysis'
FINAL_DIR.mkdir(parents=True, exist_ok=True)
print('입력:', ABL_DIR)
print('최종 산출물:', FINAL_DIR)

In [ ]:
ablation_path = ABL_DIR / 'ablation_results.csv'
baseline_path = ABL_DIR / 'baseline_last_value.csv'
if not ablation_path.exists() or not baseline_path.exists():
    raise FileNotFoundError('3단계 ablation_results.csv 및 baseline_last_value.csv 파일이 필요합니다. 저장 경로를 확인하세요.')
abl = pd.read_csv(ablation_path)
base = pd.read_csv(baseline_path)
required = ['Model','Lookback','Best Epoch','Best Val MSE (scaled)','Train Seconds',
            'MAE (°C)','MSE (°C²)','RMSE (°C)','R2','60min RMSE (°C)']
missing = [c for c in required if c not in abl.columns]
if missing:
    raise ValueError(f'3단계 결과에서 필요한 컬럼이 없습니다: {missing}\n현재 컬럼: {abl.columns.tolist()}')
assert len(abl) == 9 and set(abl['Lookback']) == {48,96,192}, '9개 실험 또는 Lookback 구성을 확인하세요.'
assert set(abl['Model']) == {'RNN','LSTM','GRU'}
assert 'RMSE (°C)' in base.columns
baseline_rmse = float(base.iloc[0]['RMSE (°C)'])
print('Ablation:',len(abl),'건 | Last Value RMSE:',round(baseline_rmse,4),'°C')
display(abl.sort_values(['Model','Lookback']).round(4))

In [ ]:
ranking = abl.copy()
ranking['Last Value 대비 RMSE 개선율 (%)'] = (baseline_rmse-ranking['RMSE (°C)'])/baseline_rmse*100
ranking = ranking.sort_values('RMSE (°C)').reset_index(drop=True)
display(ranking[['Model','Lookback','MAE (°C)','MSE (°C²)','RMSE (°C)','R2','60min RMSE (°C)',
                 'Train Seconds','Last Value 대비 RMSE 개선율 (%)']].round(4))
ranking.to_csv(FINAL_DIR/'final_model_ranking.csv',index=False,encoding='utf-8-sig')

best = ranking.iloc[0]
print(f"전체 최저 RMSE: {best['Model']} / Lookback {int(best['Lookback'])} / {best['RMSE (°C)']:.4f}°C")
print(f"기준선 대비 개선율: {best['Last Value 대비 RMSE 개선율 (%)']:.3f}%")

In [ ]:
piv = abl.pivot(index='Model',columns='Lookback',values='RMSE (°C)').reindex(['RNN','LSTM','GRU'])
display(piv.round(4))
piv.to_csv(FINAL_DIR/'final_lookback_rmse.csv',encoding='utf-8-sig')
fig,ax=plt.subplots(figsize=(8,5))
for kind in ['RNN','LSTM','GRU']:
    r=abl[abl.Model==kind].sort_values('Lookback')
    ax.plot(r.Lookback,r['RMSE (°C)'],marker='o',label=kind)
ax.axhline(baseline_rmse,ls='--',color='gray',label='Last Value')
ax.set(xticks=[48,96,192],xlabel='Lookback (15-minute steps)',ylabel='Test RMSE (°C)',
       title='ETTm1: Lookback Ablation (Horizon=4)')
ax.grid(alpha=.25);ax.legend();fig.tight_layout()
fig.savefig(FINAL_DIR/'final_lookback_comparison.png',dpi=180)
plt.show()

In [ ]:
cols=['MAE (°C)','MSE (°C²)','RMSE (°C)','60min RMSE (°C)']
rows=[]
for metric in cols:
    r=abl.loc[abl[metric].idxmin()]
    rows.append({'Metric':metric,'Best Model':r['Model'],'Lookback':int(r['Lookback']),'Value':r[metric]})
r=abl.loc[abl['R2'].idxmax()]
rows.append({'Metric':'R2','Best Model':r['Model'],'Lookback':int(r['Lookback']),'Value':r['R2']})
metric_winners=pd.DataFrame(rows)
display(metric_winners.round(4))
metric_winners.to_csv(FINAL_DIR/'final_metric_winners.csv',index=False,encoding='utf-8-sig')
fig,ax=plt.subplots(figsize=(8,5))
for kind in ['RNN','LSTM','GRU']:
    r=abl[abl.Model==kind].sort_values('Lookback')
    ax.plot(r.Lookback,r['60min RMSE (°C)'],marker='o',label=kind)
ax.set(xticks=[48,96,192],xlabel='Lookback (15-minute steps)',ylabel='60-minute-ahead RMSE (°C)',
       title='60-minute-ahead prediction error')
ax.grid(alpha=.25);ax.legend();fig.tight_layout()
fig.savefig(FINAL_DIR/'final_60min_rmse.png',dpi=180)
plt.show()

In [ ]:
fig,ax=plt.subplots(figsize=(8,5))
for kind in ['RNN','LSTM','GRU']:
    r=abl[abl.Model==kind]
    ax.scatter(r['Train Seconds'],r['RMSE (°C)'],label=kind,s=65)
    for _,q in r.iterrows():
        ax.annotate(f"L={int(q['Lookback'])}",(q['Train Seconds'],q['RMSE (°C)']),xytext=(4,5),textcoords='offset points',fontsize=8)
ax.set(xlabel='Training time (seconds)',ylabel='Test RMSE (°C)',title='Training cost vs prediction error')
ax.grid(alpha=.25);ax.legend();fig.tight_layout()
fig.savefig(FINAL_DIR/'final_time_vs_rmse.png',dpi=180)
plt.show()

In [ ]:
from IPython.display import Image
candidate_paths = [RESULT_DIR/'test_prediction_60min.png',
                   Path('/content/ettm1_results/test_prediction_60min.png'),
                   ABL_DIR/'ablation_rmse.png',ABL_DIR/'ablation_val_loss.png']
for p in candidate_paths:
    if p.exists():
        print('파일:',p)
        display(Image(filename=str(p),width=900))
    else:
        print('파일 없음(건너뜀):',p)

In [ ]:
best_by_model = (abl.loc[abl.groupby('Model')['RMSE (°C)'].idxmin()]
                   .set_index('Model').loc[['RNN','LSTM','GRU']])
summary = f"""# ETTm1 최종 결과 분석

- 데이터: ETTm1, 15분 간격, 입력 변수 7개, 예측 변수 OT.
- 실험: RNN/LSTM/GRU, Lookback 48·96·192, Horizon 4(미래 15·30·45·60분).
- 최종 평가: 공통 Test 예측 시점에서 4개 Horizon 전체를 평가한 RMSE 등.
- 전체 최저 RMSE: {best['Model']} / Lookback {int(best['Lookback'])} / {best['RMSE (°C)']:.4f}℃.
- 최저 RMSE 조건의 MAE: {best['MAE (°C)']:.4f}℃, MSE: {best['MSE (°C²)']:.4f}℃², R²: {best['R2']:.4f}.
- Last Value RMSE: {baseline_rmse:.4f}℃; 최저 RMSE 모델 개선율: {best['Last Value 대비 RMSE 개선율 (%)']:.3f}%.
- 모델별 최적 Lookback: """ + ', '.join(
    f"{kind}={int(best_by_model.loc[kind,'Lookback'])} (RMSE {best_by_model.loc[kind,'RMSE (°C)']:.4f}℃)"
    for kind in ['RNN','LSTM','GRU']) + """.

## 해석 및 한계

입력 길이가 길수록 반드시 정확도가 높아지는 것은 아니었습니다. Last Value와의 차이가 작으므로 딥러닝 모델의 뚜렷한 우위가 입증되었다고 해석하기 어렵습니다. 현재 결과는 단일 시드 실행이므로 반복 시드에 따른 변동성 검증이 필요합니다. 또한 이 결과만으로 기울기 소실이나 모델 구조의 인과적 효과를 확정할 수 없습니다.
"""
(FINAL_DIR/'final_report_summary.md').write_text(summary,encoding='utf-8')
display(Markdown(summary))

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

RESULT_DIR = Path(
    '/content/drive/MyDrive/CBNU/어프렌티스 프로젝트/ettm1_results'
)
ABL_DIR = RESULT_DIR / 'ablation'

abl = pd.read_csv(ABL_DIR / 'ablation_results.csv')
baseline = pd.read_csv(ABL_DIR / 'baseline_last_value.csv')

cols = ['Model', 'MAE (°C)', 'MSE (°C²)', 'RMSE (°C)', 'R2']

basic_96 = (
    abl.loc[abl['Lookback'] == 96, cols]
    .set_index('Model')
    .loc[['RNN', 'LSTM', 'GRU']]
    .reset_index()
)


basic_table = pd.concat(
    [basic_96, baseline[cols]],
    ignore_index=True
)

print('Lookback 96: 공통 테스트 시점 기준')
display(basic_table.round(4))


basic_table.to_csv(
    RESULT_DIR / 'test_metrics_aligned_96.csv',
    index=False,
    encoding='utf-8-sig'
)

In [ ]:

from pathlib import Path
import copy, time, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import mean_absolute_error, mean_squared_error
from IPython.display import display

REVIEW_DIR = OUT_DIR / 'reviewer_validation'
REVIEW_DIR.mkdir(parents=True, exist_ok=True)
REVIEW_SEEDS = [42, 123, 2026]
REVIEW_LAYERS = [1, 2, 3]
REVIEW_KINDS = ['RNN', 'LSTM', 'GRU']
REVIEW_LB, REVIEW_MAX_LB = 96, 192
REVIEW_DROPOUT = 0.0


review_loaders = {}
for part in ['train', 'val', 'test']:
    ds = ETTWindowDataset(scaled[part], lookback=REVIEW_LB, horizon=HORIZON, target_idx=target_idx)
    aligned = Subset(ds, range(REVIEW_MAX_LB - REVIEW_LB, len(ds)))
    assert len(aligned) > 0
    review_loaders[part] = DataLoader(aligned, batch_size=BATCH_SIZE, shuffle=(part == 'train'))
print('공통 평가 샘플 수:', {k:len(v.dataset) for k,v in review_loaders.items()})
print('모든 검증 실험: Lookback=96, Horizon=', HORIZON, ', Dropout=', REVIEW_DROPOUT)


In [ ]:
def review_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def review_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total = count = 0
    for x,y in loader:
        x,y = x.to(DEVICE), y.to(DEVICE)
        if training: optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            loss = torch.nn.functional.mse_loss(model(x),y)
            if training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
        total += loss.item()*len(x); count += len(x)
    return total/count

@torch.no_grad()
def review_predict(model, loader):
    model.eval(); ps=[]; ys=[]
    for x,y in loader:
        ps.append(model(x.to(DEVICE)).cpu().numpy()); ys.append(y.numpy())
    return inverse_ot(np.concatenate(ps)), inverse_ot(np.concatenate(ys))

review_rows=[]
for layers in REVIEW_LAYERS:
    for kind in REVIEW_KINDS:
        for seed in REVIEW_SEEDS:
            review_seed(seed)
            model=ForecastRNN(kind, INPUT_SIZE, HIDDEN_SIZE, layers, REVIEW_DROPOUT, HORIZON).to(DEVICE)
            params=sum(p.numel() for p in model.parameters())
            opt=torch.optim.Adam(model.parameters(),lr=LR)
            best=float('inf'); wait=0; best_ep=0; best_state=None
            start=time.perf_counter()
            for ep in range(1,MAX_EPOCHS+1):
                train_loss=review_epoch(model,review_loaders['train'],opt)
                val_loss=review_epoch(model,review_loaders['val'])
                if val_loss < best-MIN_DELTA:
                    best=val_loss; best_ep=ep; best_state=copy.deepcopy(model.state_dict()); wait=0
                else:
                    wait+=1
                    if wait>=PATIENCE: break
            seconds=time.perf_counter()-start
            model.load_state_dict(best_state)
            pred,true=review_predict(model,review_loaders['test'])
            row={'Model':kind,'Layers':layers,'Seed':seed,'Params':params,
                 'Best Epoch':best_ep,'Val MSE (scaled)':best,'Train Seconds':seconds,
                 'MAE (°C)':mean_absolute_error(true.ravel(),pred.ravel()),
                 'RMSE (°C)':np.sqrt(mean_squared_error(true.ravel(),pred.ravel())),
                 '60min RMSE (°C)':np.sqrt(mean_squared_error(true[:,-1],pred[:,-1]))}
            review_rows.append(row)
            print(f'{kind:4} layers={layers} seed={seed} RMSE={row["RMSE (°C)"]:.4f} time={seconds:.1f}s')
            pd.DataFrame(review_rows).to_csv(REVIEW_DIR/'review_raw.csv',index=False,encoding='utf-8-sig')
            del model
            if torch.cuda.is_available(): torch.cuda.empty_cache()

review_df=pd.DataFrame(review_rows)
review_summary=review_df.groupby(['Model','Layers'],as_index=False).agg(
    RMSE_mean=('RMSE (°C)','mean'),RMSE_std=('RMSE (°C)','std'),
    MAE_mean=('MAE (°C)','mean'),Time_mean=('Train Seconds','mean'),
    Params=('Params','first'))
review_summary.to_csv(REVIEW_DIR/'review_summary.csv',index=False,encoding='utf-8-sig')
display(review_summary.round(4))
fig,ax=plt.subplots(figsize=(8,5))
for kind in REVIEW_KINDS:
    r=review_summary[review_summary.Model==kind].sort_values('Layers')
    ax.errorbar(r.Layers,r.RMSE_mean,yerr=r.RMSE_std.fillna(0),marker='o',capsize=4,label=kind)
ax.set(xticks=REVIEW_LAYERS,xlabel='Number of layers',ylabel='Test RMSE (°C)',title='Layer comparison (same test targets)')
ax.legend(); ax.grid(alpha=.3); fig.tight_layout()
fig.savefig(REVIEW_DIR/'layer_rmse.png',dpi=180); plt.show()


In [ ]:
full_n=len(ETTWindowDataset(scaled['test'],lookback=96,horizon=HORIZON,target_idx=target_idx))
aligned_n=len(review_loaders['test'].dataset)
print(f'기본 Lookback=96: {full_n}개 예측 시작점')
print(f'Lookback 공통 비교: {aligned_n}개 예측 시작점 (앞 {full_n-aligned_n}개 제외)')
print('두 결과는 동일한 평가 집합이 아니며, 별도 학습에 따른 확률적 변동도 있습니다.')

base_path=OUT_DIR/'test_metrics.csv'
abl_path=OUT_DIR/'ablation'/'ablation_results.csv'
if base_path.exists() and abl_path.exists():
    basic=pd.read_csv(base_path)
    ablation=pd.read_csv(abl_path)
    b=basic[basic.Model.isin(REVIEW_KINDS)][['Model','RMSE (°C)']].rename(columns={'RMSE (°C)':'Basic RMSE'})
    a=ablation[(ablation.Lookback==96)&ablation.Model.isin(REVIEW_KINDS)][['Model','RMSE (°C)']].rename(columns={'RMSE (°C)':'Aligned RMSE'})
    comparison=b.merge(a,on='Model')
    comparison['Difference (aligned - basic)']=comparison['Aligned RMSE']-comparison['Basic RMSE']
    display(comparison.round(4))
    comparison.to_csv(REVIEW_DIR/'original_vs_aligned.csv',index=False,encoding='utf-8-sig')
else:
    print('기존 결과 CSV가 없어서 수치 비교는 생략했습니다. 기존 학습 셀을 먼저 실행하세요.')


controlled=review_summary[review_summary.Layers==2].sort_values('RMSE_mean')
print('통제 실험 (Layer=2) 순위:')
display(controlled.round(4))


In [ ]:
layer_best=review_summary.loc[review_summary.groupby('Model').RMSE_mean.idxmin()].sort_values('Model')
fixed=review_summary[review_summary.Layers==2].sort_values('RMSE_mean')
print('질문 1: 모델별 RMSE가 가장 낮았던 층 수 (이번 검증 실험 기준)')
display(layer_best[['Model','Layers','RMSE_mean','RMSE_std','Time_mean']].round(4))
print('질문 2: 기존 기본 실험과 Lookback 실험은 테스트 샘플 수가 다릅니다. 표와 그래프 비교는 동일 시점 평가 결과를 사용해야 합니다.')
print('질문 3: 동일 조건 Layer=2 모델별 성능 / 파라미터 / 시간')
display(fixed[['Model','RMSE_mean','RMSE_std','Params','Time_mean']].round(4))
print('주의: GRU의 구조적 장점은 가능한 해석이지, 본 실험만으로 성능 차이의 인과 원인을 입증하지는 못합니다.')
